In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

## 2. Load Dataset

The data transformation stage uses the dataset produced by the
Data Ingestion stage.

At this stage, we will prepare the data for machine learning while
maintaining a strict separation between training and test data.

The transformation process will include:

1. Train/test splitting
2. Missing-value handling
3. Categorical encoding
4. Numerical feature scaling
5. Construction of a reusable preprocessing pipeline

The test dataset must not influence any parameters learned during
preprocessing.

In [2]:
DATA_PATH = Path(
    "../artifacts/data_ingestion/AmesHousing.txt"
)

df = pd.read_csv(
    DATA_PATH,
    sep="\t"
)
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## 3. Separate Features and Target

The target variable for this project is `SalePrice`.

The remaining columns represent the predictor variables used to
estimate the target.

Therefore:

- `X` contains the predictor features.
- `y` contains the target variable.

In [3]:
X = df.drop(
    columns=["SalePrice"]
)

y = df["SalePrice"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (2930, 81)
Target shape: (2930,)


## 4. Train/Test Split

A machine learning model must be evaluated on observations that were
not used during training.

We therefore divide the dataset into:

- Training set — used to learn preprocessing parameters and model
  parameters.
- Test set — held back for final evaluation.

We use an 80/20 split.

The `random_state` parameter makes the split reproducible.

Most importantly, preprocessing parameters must be learned from the
training set only.

In [4]:

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2344, 81)
X_test : (586, 81)
y_train: (2344,)
y_test : (586,)


### Interpretation

The dataset has been divided into training and test subsets.

The training subset will be used to learn preprocessing parameters and
model parameters.

The test subset will remain unseen during training and preprocessing
parameter estimation.

This separation helps us obtain an unbiased estimate of how the final
model performs on unseen data.

## 5. Verify Train/Test Split

Before continuing with preprocessing, we verify that the split produced
the expected number of observations and that the target values remain
aligned with their corresponding feature rows.

In [5]:
print(
    "Training observations:",
    len(X_train)
)

print(
    "Test observations:",
    len(X_test)
)

print(
    "Total observations:",
    len(X_train) + len(X_test)
)

Training observations: 2344
Test observations: 586
Total observations: 2930


In [9]:
assert len(X_train) + len(X_test) == len(X)

assert len(y_train) == len(X_train)

assert len(y_test) == len(X_test)
print("Train/test split validation passed.")

Train/test split validation passed.


## 6. Identify Numerical and Categorical Features

Different feature types require different preprocessing strategies.

Numerical features may require:

- Missing-value imputation
- Scaling

Categorical features may require:

- Missing-value handling
- Encoding

We therefore identify the two groups before constructing the
`ColumnTransformer`.

In [10]:
numerical_features = (
    X_train
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(
        exclude=np.number
    )
    .columns
    .tolist()
)

In [11]:
print(
    "Numerical features:",
    len(numerical_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)

Numerical features: 38
Categorical features: 43


In [12]:
print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


Numerical features:
['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd', 'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold']

Categorical features:
['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Hea

## 7. Missing Values in Training Features

Missing-value handling must be based on the training data.

We first identify which numerical and categorical predictor variables
contain missing values.

The test set will not be used to learn imputation parameters.

## 8. Analyze Missing Values in the Training Features

Missing values must be investigated before selecting an imputation
strategy.

We first identify the numerical features that contain missing values.

The analysis is performed on the training data because preprocessing
parameters must be learned from the training set only.

In [14]:
numerical_missing_values = (
    X_train[numerical_features]
    .isnull()
    .sum().sort_values(ascending=False)
)

numerical_missing_values = numerical_missing_values[
    numerical_missing_values > 0
]
numerical_missing_values

Lot Frontage      393
Garage Yr Blt     122
Mas Vnr Area       19
BsmtFin SF 2        1
Bsmt Unf SF         1
Total Bsmt SF       1
Garage Cars         1
Garage Area         1
Bsmt Full Bath      1
BsmtFin SF 1        1
Bsmt Half Bath      1
dtype: int64

## 9. Missing Values in Categorical Features

We now inspect missing values in categorical features.

The percentage of missing values alone is not enough to determine the
correct treatment.

For the Ames Housing dataset, some missing categorical values represent
the absence of a physical feature rather than an unknown value.

Therefore, we will use the meaning of each feature when selecting the
imputation strategy.

In [15]:
categorical_missing_values = (
    X_train[categorical_features]
    .isnull()
    .sum().sort_values(ascending=False)
)
categorical_missing_values = categorical_missing_values[
    categorical_missing_values > 0
]
categorical_missing_values


Pool QC           2332
Misc Feature      2250
Alley             2182
Fence             1874
Mas Vnr Type      1426
Fireplace Qu      1144
Garage Cond        122
Garage Qual        122
Garage Finish      122
Garage Type        120
Bsmt Exposure       63
BsmtFin Type 2      62
BsmtFin Type 1      61
Bsmt Qual           61
Bsmt Cond           61
dtype: int64

## 10. Missing-Value Percentage

The number of missing observations does not tell us the relative
severity of missingness.

For example, 500 missing values represent a very different situation
in a dataset containing 1,000 rows compared with a dataset containing
100,000 rows.

We therefore calculate the percentage of missing values for each
feature.

In [17]:
missing_summary = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_percentage": (
        X_train.isna()
        .mean()
        .mul(100)
    )
})

missing_summary = (
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
    .sort_values(
        "missing_percentage",
        ascending=False
    )
)

missing_summary

,missing_count,missing_percentage
Pool QC,2332,99.488055
Misc Feature,2250,95.989761
Alley,2182,93.088737
Fence,1874,79.948805
Mas Vnr Type,1426,60.836177
Fireplace Qu,1144,48.805461
Lot Frontage,393,16.766212
Garage Cond,122,5.204778
Garage Yr Blt,122,5.204778
Garage Finish,122,5.204778


## 11. Missing-Value Meaning

Missing values can have different meanings.

A missing value may indicate:

1. A physical feature does not exist.
2. Information was not recorded.
3. The value is genuinely unknown.
4. The value is not applicable.

We should not automatically treat every missing value in the same way.

The Ames Housing dataset contains several categorical features where
missingness can represent the absence of a property feature.

We therefore investigate the affected features before choosing an
imputation strategy.

In [18]:
missing_summary

,missing_count,missing_percentage
Pool QC,2332,99.488055
Misc Feature,2250,95.989761
Alley,2182,93.088737
Fence,1874,79.948805
Mas Vnr Type,1426,60.836177
Fireplace Qu,1144,48.805461
Lot Frontage,393,16.766212
Garage Cond,122,5.204778
Garage Yr Blt,122,5.204778
Garage Finish,122,5.204778


## 11.1 Inspect Categorical Missing Values

For each categorical feature containing missing values, we inspect the
observed categories.

This helps us determine whether the missing value likely represents
absence of the feature or an unknown/unrecorded value.

In [20]:
for column in categorical_missing_values.index:
    
    print("=" * 60)
    print(f"Feature: {column}")
    print("=" * 60)
    
    print(
        X_train[column]
        .value_counts(dropna=False)
    )
    
    print()

Feature: Pool QC
Pool QC
NaN    2332
Ex        4
Gd        4
TA        3
Fa        1
Name: count, dtype: int64

Feature: Misc Feature
Misc Feature
NaN     2250
Shed      84
Gar2       5
Othr       4
Elev       1
Name: count, dtype: int64

Feature: Alley
Alley
NaN     2182
Grvl     100
Pave      62
Name: count, dtype: int64

Feature: Fence
Fence
NaN      1874
MnPrv     273
GdPrv      95
GdWo       91
MnWw       11
Name: count, dtype: int64

Feature: Mas Vnr Type
Mas Vnr Type
NaN        1426
BrkFace     714
Stone       182
BrkCmn       21
CBlock        1
Name: count, dtype: int64

Feature: Fireplace Qu
Fireplace Qu
NaN    1144
Gd      580
TA      483
Fa       64
Po       38
Ex       35
Name: count, dtype: int64

Feature: Garage Cond
Garage Cond
TA     2134
NaN     122
Fa       60
Gd       13
Po       12
Ex        3
Name: count, dtype: int64

Feature: Garage Qual
Garage Qual
TA     2093
NaN     122
Fa      103
Gd       18
Po        5
Ex        3
Name: count, dtype: int64

Feature: Garage 

In [21]:
print(
    "Pool Area = 0:",
    (X_train["Pool Area"] == 0).sum()
)

print(
    "Pool Area > 0:",
    (X_train["Pool Area"] > 0).sum()
)

Pool Area = 0: 2332
Pool Area > 0: 12


In [22]:
print(
    pd.crosstab(
        X_train["Pool Area"] == 0,
        X_train["Pool QC"].isna()
    )
)

Pool QC    False  True 
Pool Area              
False         12      0
True           0   2332


In [23]:
absence_features = [
    "Alley",
    "Bsmt Qual",
    "Bsmt Cond",
    "Bsmt Exposure",
    "BsmtFin Type 1",
    "BsmtFin Type 2",
    "Fireplace Qu",
    "Garage Type",
    "Garage Finish",
    "Garage Qual",
    "Garage Cond",
    "Pool QC",
    "Fence",
    "Misc Feature",
]

In [24]:
absence_features = [
    column
    for column in absence_features
    if column in X_train.columns
]

print(absence_features)

['Alley', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond', 'Pool QC', 'Fence', 'Misc Feature']


In [25]:
for column in absence_features:
    
    print(
        f"{column}: "
        f"{X_train[column].isna().sum()} missing values"
    )

Alley: 2182 missing values
Bsmt Qual: 61 missing values
Bsmt Cond: 61 missing values
Bsmt Exposure: 63 missing values
BsmtFin Type 1: 61 missing values
BsmtFin Type 2: 62 missing values
Fireplace Qu: 1144 missing values
Garage Type: 120 missing values
Garage Finish: 122 missing values
Garage Qual: 122 missing values
Garage Cond: 122 missing values
Pool QC: 2332 missing values
Fence: 1874 missing values
Misc Feature: 2250 missing values


In [26]:
categorical_absence_features = [
    "Alley",
    "Bsmt Qual",
    "Bsmt Cond",
    "Bsmt Exposure",
    "BsmtFin Type 1",
    "BsmtFin Type 2",
    "Fireplace Qu",
    "Garage Type",
    "Garage Finish",
    "Garage Qual",
    "Garage Cond",
    "Pool QC",
    "Fence",
    "Misc Feature",
]

In [27]:
print(
    X_train[
        categorical_absence_features
    ].dtypes
)

Alley             str
Bsmt Qual         str
Bsmt Cond         str
Bsmt Exposure     str
BsmtFin Type 1    str
BsmtFin Type 2    str
Fireplace Qu      str
Garage Type       str
Garage Finish     str
Garage Qual       str
Garage Cond       str
Pool QC           str
Fence             str
Misc Feature      str
dtype: object


In [30]:
categorical_missing_features = (
    categorical_missing_values.index.tolist()
)

print(
    "Categorical features with missing values:"
)

for feature in categorical_missing_features:
    print(feature)

Categorical features with missing values:
Pool QC
Misc Feature
Alley
Fence
Mas Vnr Type
Fireplace Qu
Garage Cond
Garage Qual
Garage Finish
Garage Type
Bsmt Exposure
BsmtFin Type 2
BsmtFin Type 1
Bsmt Qual
Bsmt Cond


In [31]:
missing_investigation = pd.DataFrame(
    {
        "feature": categorical_missing_features,
        "missing_count": [
            X_train[feature].isna().sum()
            for feature in categorical_missing_features
        ],
        "missing_percentage": [
            X_train[feature].isna().mean() * 100
            for feature in categorical_missing_features
        ],
        "possible_meaning": "",
        "evidence": "",
        "decision": "",
    }
)

missing_investigation

,feature,missing_count,missing_percentage,possible_meaning,evidence,decision
0,Pool QC,2332,99.488055,,,
1,Misc Feature,2250,95.989761,,,
2,Alley,2182,93.088737,,,
3,Fence,1874,79.948805,,,
4,Mas Vnr Type,1426,60.836177,,,
5,Fireplace Qu,1144,48.805461,,,
6,Garage Cond,122,5.204778,,,
7,Garage Qual,122,5.204778,,,
8,Garage Finish,122,5.204778,,,
9,Garage Type,120,5.119454,,,


In [32]:
print(
    X_train["Pool QC"]
    .value_counts(dropna=False)
)

Pool QC
NaN    2332
Ex        4
Gd        4
TA        3
Fa        1
Name: count, dtype: int64


In [33]:
print(
    X_train["Pool Area"]
    .describe()
)

count    2344.000000
mean        2.582765
std        38.324145
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       800.000000
Name: Pool Area, dtype: float64


In [34]:
print(
    "Pool Area = 0:",
    (X_train["Pool Area"] == 0).sum()
)

print(
    "Pool Area > 0:",
    (X_train["Pool Area"] > 0).sum()
)

Pool Area = 0: 2332
Pool Area > 0: 12


In [35]:
pool_check = pd.crosstab(
    X_train["Pool Area"] == 0,
    X_train["Pool QC"].isna(),
    rownames=["Pool Area = 0"],
    colnames=["Pool QC Missing"]
)

pool_check

Pool QC Missing,False,True
Pool Area = 0,,
False,12,0
True,0,2332


### Finding — Pool QC

`Pool QC` contains missing values.

The related `Pool Area` feature was examined to determine whether
missing `Pool QC` values correspond to properties without a pool.

The missing-value treatment will therefore be based on the observed
relationship between these two features rather than assuming that every
missing categorical value means "unknown".

In [36]:
print(
    X_train["Garage Type"]
    .value_counts(dropna=False)
)

Garage Type
Attchd     1397
Detchd      631
BuiltIn     138
NaN         120
Basment      30
2Types       18
CarPort      10
Name: count, dtype: int64


In [37]:
print(
    X_train["Garage Area"]
    .describe()
)

count    2343.000000
mean      469.078959
std       212.432786
min         0.000000
25%       319.000000
50%       476.000000
75%       576.000000
max      1488.000000
Name: Garage Area, dtype: float64


In [38]:
print(
    "Garage Area = 0:",
    (X_train["Garage Area"] == 0).sum()
)

print(
    "Garage Area > 0:",
    (X_train["Garage Area"] > 0).sum()
)

Garage Area = 0: 120
Garage Area > 0: 2223


In [39]:
garage_check = pd.crosstab(
    X_train["Garage Area"] == 0,
    X_train["Garage Type"].isna(),
    rownames=["Garage Area = 0"],
    colnames=["Garage Type Missing"]
)

garage_check

Garage Type Missing,False,True
Garage Area = 0,,
False,2224,0
True,0,120


In [40]:
print(
    X_train["Bsmt Qual"]
    .value_counts(dropna=False)
)

Bsmt Qual
TA     1057
Gd      970
Ex      187
Fa       67
NaN      61
Po        2
Name: count, dtype: int64


In [41]:
print(
    X_train["Total Bsmt SF"]
    .describe()
)

count    2343.000000
mean     1047.022194
std       436.567117
min         0.000000
25%       784.000000
50%       988.000000
75%      1288.000000
max      6110.000000
Name: Total Bsmt SF, dtype: float64


In [42]:
print(
    "Total Bsmt SF = 0:",
    (X_train["Total Bsmt SF"] == 0).sum()
)

print(
    "Total Bsmt SF > 0:",
    (X_train["Total Bsmt SF"] > 0).sum()
)

Total Bsmt SF = 0: 60
Total Bsmt SF > 0: 2283


In [43]:
basement_check = pd.crosstab(
    X_train["Total Bsmt SF"] == 0,
    X_train["Bsmt Qual"].isna(),
    rownames=["Total Bsmt SF = 0"],
    colnames=["Bsmt Qual Missing"]
)

basement_check

Bsmt Qual Missing,False,True
Total Bsmt SF = 0,,
False,2283,1
True,0,60


In [44]:
print(
    X_train["Fireplace Qu"]
    .value_counts(dropna=False)
)

Fireplace Qu
NaN    1144
Gd      580
TA      483
Fa       64
Po       38
Ex       35
Name: count, dtype: int64


In [45]:
print(
    X_train["Fireplaces"]
    .value_counts(dropna=False)
)

Fireplaces
0    1144
1    1009
2     179
3      11
4       1
Name: count, dtype: int64


In [46]:
fireplace_check = pd.crosstab(
    X_train["Fireplaces"] == 0,
    X_train["Fireplace Qu"].isna(),
    rownames=["Fireplaces = 0"],
    colnames=["Fireplace Qu Missing"]
)

fireplace_check

Fireplace Qu Missing,False,True
Fireplaces = 0,,
False,1200,0
True,0,1144


## 12. Numerical Preprocessing Pipeline

Numerical features require two primary preprocessing operations in our
baseline pipeline:

1. Missing-value imputation
2. Feature scaling

For missing numerical values, we use median imputation because the
median is less sensitive to extreme values than the mean.

After imputation, numerical features are standardized using
`StandardScaler`.

The preprocessing parameters will be learned from the training data
only.

In [47]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [48]:
numerical_pipeline=Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [49]:
numerical_pipeline.fit(
    X_train[numerical_features]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](38,)","['Order','PID','MS SubClass',...,'Misc Val','Mo Sold','Yr Sold']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,38
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"co

In [51]:
numerical_pipeline.named_steps[
    "imputer"
].statistics_

array([1.4660000e+03, 5.3545406e+08, 5.0000000e+01, 6.8000000e+01,
       9.3565000e+03, 6.0000000e+00, 5.0000000e+00, 1.9720000e+03,
       1.9920000e+03, 0.0000000e+00, 3.7500000e+02, 0.0000000e+00,
       4.6200000e+02, 9.8800000e+02, 1.0820000e+03, 0.0000000e+00,
       0.0000000e+00, 1.4365000e+03, 0.0000000e+00, 0.0000000e+00,
       2.0000000e+00, 0.0000000e+00, 3.0000000e+00, 1.0000000e+00,
       6.0000000e+00, 1.0000000e+00, 1.9780000e+03, 2.0000000e+00,
       4.7600000e+02, 0.0000000e+00, 2.6000000e+01, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       6.0000000e+00, 2.0080000e+03])

In [50]:
numerical_pipeline.named_steps

{'imputer': SimpleImputer(strategy='median'), 'scaler': StandardScaler()}

In [52]:
numerical_imputation_values = pd.Series(
    numerical_pipeline
    .named_steps["imputer"]
    .statistics_,
    index=numerical_features,
    name="imputation_value"
)

numerical_imputation_values

Order                   1466.0
PID                535454060.0
MS SubClass               50.0
Lot Frontage              68.0
Lot Area                9356.5
Overall Qual               6.0
Overall Cond               5.0
Year Built              1972.0
Year Remod/Add          1992.0
Mas Vnr Area               0.0
BsmtFin SF 1             375.0
BsmtFin SF 2               0.0
Bsmt Unf SF              462.0
Total Bsmt SF            988.0
1st Flr SF              1082.0
2nd Flr SF                 0.0
Low Qual Fin SF            0.0
Gr Liv Area             1436.5
Bsmt Full Bath             0.0
Bsmt Half Bath             0.0
Full Bath                  2.0
Half Bath                  0.0
Bedroom AbvGr              3.0
Kitchen AbvGr              1.0
TotRms AbvGrd              6.0
Fireplaces                 1.0
Garage Yr Blt           1978.0
Garage Cars                2.0
Garage Area              476.0
Wood Deck SF               0.0
Open Porch SF             26.0
Enclosed Porch             0.0
3Ssn Por

In [53]:
X_train_numerical = (
    numerical_pipeline.transform(
        X_train[numerical_features]
    )
)

In [54]:
print(
    "Transformed numerical shape:",
    X_train_numerical.shape
)

Transformed numerical shape: (2344, 38)


In [55]:
X_test_numerical = (
    numerical_pipeline.transform(
        X_test[numerical_features]
    )
)

In [56]:
categorical_other_features = [
    column
    for column in categorical_features
    if column not in categorical_absence_features
]

In [57]:
print(
    "Absence categorical features:",
    len(categorical_absence_features)
)

print(
    "Other categorical features:",
    len(categorical_other_features)
)

print(
    "Total categorical features:",
    len(categorical_features)
)

Absence categorical features: 14
Other categorical features: 29
Total categorical features: 43


In [58]:
assert (
    len(categorical_absence_features)
    + len(categorical_other_features)
    == len(categorical_features)
)
print(
    "Categorical feature grouping validated."
)

Categorical feature grouping validated.


In [60]:
from sklearn.preprocessing import OneHotEncoder

In [61]:
categorical_absence_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ]
)

In [62]:
categorical_other_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ]
)

In [65]:
X_train_absence = (
    categorical_absence_pipeline.fit_transform(
        X_train[categorical_absence_features]
    )
)

In [66]:
print(
    "Absence categorical transformed shape:",
    X_train_absence.shape
)

Absence categorical transformed shape: (2344, 78)


In [67]:
X_train_other = (
    categorical_other_pipeline.fit_transform(
        X_train[categorical_other_features]
    )
)

In [68]:
print(
    "Other categorical transformed shape:",
    X_train_other.shape
)

Other categorical transformed shape: (2344, 200)


In [69]:
absence_encoder = (
    categorical_absence_pipeline
    .named_steps["encoder"]
)
absence_encoder.categories_

[array(['Grvl', 'None', 'Pave'], dtype=object),
 array(['Ex', 'Fa', 'Gd', 'None', 'Po', 'TA'], dtype=object),
 array(['Ex', 'Fa', 'Gd', 'None', 'Po', 'TA'], dtype=object),
 array(['Av', 'Gd', 'Mn', 'No', 'None'], dtype=object),
 array(['ALQ', 'BLQ', 'GLQ', 'LwQ', 'None', 'Rec', 'Unf'], dtype=object),
 array(['ALQ', 'BLQ', 'GLQ', 'LwQ', 'None', 'Rec', 'Unf'], dtype=object),
 array(['Ex', 'Fa', 'Gd', 'None', 'Po', 'TA'], dtype=object),
 array(['2Types', 'Attchd', 'Basment', 'BuiltIn', 'CarPort', 'Detchd',
        'None'], dtype=object),
 array(['Fin', 'None', 'RFn', 'Unf'], dtype=object),
 array(['Ex', 'Fa', 'Gd', 'None', 'Po', 'TA'], dtype=object),
 array(['Ex', 'Fa', 'Gd', 'None', 'Po', 'TA'], dtype=object),
 array(['Ex', 'Fa', 'Gd', 'None', 'TA'], dtype=object),
 array(['GdPrv', 'GdWo', 'MnPrv', 'MnWw', 'None'], dtype=object),
 array(['Elev', 'Gar2', 'None', 'Othr', 'Shed'], dtype=object)]

In [70]:
other_encoder = (
    categorical_other_pipeline
    .named_steps["encoder"]
)

other_encoder.categories_

[array(['A (agr)', 'C (all)', 'FV', 'I (all)', 'RH', 'RL', 'RM'],
       dtype=object),
 array(['Grvl', 'Pave'], dtype=object),
 array(['IR1', 'IR2', 'IR3', 'Reg'], dtype=object),
 array(['Bnk', 'HLS', 'Low', 'Lvl'], dtype=object),
 array(['AllPub', 'NoSeWa', 'NoSewr'], dtype=object),
 array(['Corner', 'CulDSac', 'FR2', 'FR3', 'Inside'], dtype=object),
 array(['Gtl', 'Mod', 'Sev'], dtype=object),
 array(['Blmngtn', 'Blueste', 'BrDale', 'BrkSide', 'ClearCr', 'CollgCr',
        'Crawfor', 'Edwards', 'Gilbert', 'Greens', 'GrnHill', 'IDOTRR',
        'Landmrk', 'MeadowV', 'Mitchel', 'NAmes', 'NPkVill', 'NWAmes',
        'NoRidge', 'NridgHt', 'OldTown', 'SWISU', 'Sawyer', 'SawyerW',
        'Somerst', 'StoneBr', 'Timber', 'Veenker'], dtype=object),
 array(['Artery', 'Feedr', 'Norm', 'PosA', 'PosN', 'RRAe', 'RRAn', 'RRNe',
        'RRNn'], dtype=object),
 array(['Artery', 'Feedr', 'Norm', 'PosA', 'PosN', 'RRAe', 'RRAn', 'RRNn'],
       dtype=object),
 array(['1Fam', '2fmCon', 'Duplex', 'Twnh

In [72]:
absence_feature_names = (
    categorical_absence_pipeline
    .named_steps["encoder"]
    .get_feature_names_out(
        categorical_absence_features
    )
)

print(
    "Generated absence features:",
    len(absence_feature_names)
)

Generated absence features: 78


In [ ]:
print(
    absence_feature_names
)

In [74]:
other_feature_names = (
    categorical_other_pipeline
    .named_steps["encoder"]
    .get_feature_names_out(
        categorical_other_features
    )
)

print(
    "Generated other categorical features:",
    len(other_feature_names)
)

Generated other categorical features: 200


### Interpretation

Categorical preprocessing is divided into two groups based on the
meaning of missing values.

For categorical features where missingness represents the absence of a
property component, missing values are replaced with the explicit
category `"None"`.

Other categorical features use most-frequent imputation as the initial
baseline.

Both groups are then processed using one-hot encoding.

`handle_unknown="ignore"` ensures that categories not observed during
training do not cause the preprocessing pipeline to fail during
inference.

The preprocessing components are fitted using training data only.

## 14. ColumnTransformer

The dataset contains different types of features that require different
preprocessing operations.

`ColumnTransformer` allows us to apply separate preprocessing pipelines
to different groups of columns.

In our preprocessing architecture:

- Numerical features use the numerical pipeline.
- Categorical features where missingness represents absence use the
  categorical absence pipeline.
- Other categorical features use the categorical other pipeline.

The transformed outputs are then combined into one feature matrix that
can be provided to a machine learning model.

In [75]:
from sklearn.compose import ColumnTransformer

In [76]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical_absence",
            categorical_absence_pipeline,
            categorical_absence_features
        ),
        (
            "categorical_other",
            categorical_other_pipeline,
            categorical_other_features
        ),
    ]
)

In [77]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical_absence', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_

In [78]:
assigned_features = (
    numerical_features
    + categorical_absence_features
    + categorical_other_features
)

print(
    "Original feature count:",
    len(X_train.columns)
)

print(
    "Assigned feature count:",
    len(assigned_features)
)

Original feature count: 81
Assigned feature count: 81


In [79]:
assert set(assigned_features) == set(
    X_train.columns
)

print(
    "All features are assigned to a preprocessing group."
)

All features are assigned to a preprocessing group.


In [80]:
from collections import Counter

feature_counts = Counter(
    assigned_features
)

duplicate_features = [
    feature
    for feature, count
    in feature_counts.items()
    if count > 1
]

print(
    "Duplicate feature assignments:",
    duplicate_features
)

Duplicate feature assignments: []


In [81]:
assert not duplicate_features

In [82]:
X_train_transformed = (
    preprocessor.fit_transform(X_train)
)

In [83]:
print(
    "Original X_train shape:",
    X_train.shape
)

print(
    "Transformed X_train shape:",
    X_train_transformed.shape
)

Original X_train shape: (2344, 81)
Transformed X_train shape: (2344, 316)


In [84]:
X_test_transformed = (
    preprocessor.transform(X_test)
)

In [85]:
print(
    "Original X_test shape:",
    X_test.shape
)

print(
    "Transformed X_test shape:",
    X_test_transformed.shape
)

Original X_test shape: (586, 81)
Transformed X_test shape: (586, 316)


In [86]:
assert X_train_transformed.shape[0] == X_train.shape[0]
assert X_test_transformed.shape[0] == X_test.shape[0]

assert (
    X_train_transformed.shape[1]
    == X_test_transformed.shape[1]
)

print(
    "Transformed train/test shapes are compatible."
)

Transformed train/test shapes are compatible.


In [87]:
import numpy as np

train_array = (
    X_train_transformed.toarray()
    if hasattr(
        X_train_transformed,
        "toarray"
    )
    else X_train_transformed
)

test_array = (
    X_test_transformed.toarray()
    if hasattr(
        X_test_transformed,
        "toarray"
    )
    else X_test_transformed
)

print(
    "NaN values in transformed train:",
    np.isnan(train_array).sum()
)

print(
    "NaN values in transformed test:",
    np.isnan(test_array).sum()
)

NaN values in transformed train: 0
NaN values in transformed test: 0


In [88]:
print(
    "Infinite values in transformed train:",
    np.isinf(train_array).sum()
)

print(
    "Infinite values in transformed test:",
    np.isinf(test_array).sum()
)

Infinite values in transformed train: 0
Infinite values in transformed test: 0


In [90]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)
print(
    "Number of generated features:",
    len(feature_names)
)
print(feature_names)

Number of generated features: 316
['numerical__Order' 'numerical__PID' 'numerical__MS SubClass'
 'numerical__Lot Frontage' 'numerical__Lot Area' 'numerical__Overall Qual'
 'numerical__Overall Cond' 'numerical__Year Built'
 'numerical__Year Remod/Add' 'numerical__Mas Vnr Area'
 'numerical__BsmtFin SF 1' 'numerical__BsmtFin SF 2'
 'numerical__Bsmt Unf SF' 'numerical__Total Bsmt SF'
 'numerical__1st Flr SF' 'numerical__2nd Flr SF'
 'numerical__Low Qual Fin SF' 'numerical__Gr Liv Area'
 'numerical__Bsmt Full Bath' 'numerical__Bsmt Half Bath'
 'numerical__Full Bath' 'numerical__Half Bath' 'numerical__Bedroom AbvGr'
 'numerical__Kitchen AbvGr' 'numerical__TotRms AbvGrd'
 'numerical__Fireplaces' 'numerical__Garage Yr Blt'
 'numerical__Garage Cars' 'numerical__Garage Area'
 'numerical__Wood Deck SF' 'numerical__Open Porch SF'
 'numerical__Enclosed Porch' 'numerical__3Ssn Porch'
 'numerical__Screen Porch' 'numerical__Pool Area' 'numerical__Misc Val'
 'numerical__Mo Sold' 'numerical__Yr Sold'
 '

In [91]:
X_train_transformed_df = pd.DataFrame(
    train_array,
    columns=feature_names,
    index=X_train.index
)

In [92]:
X_train_transformed_df.head()

,numerical__Order,numerical__PID,numerical__MS SubClass,numerical__Lot Frontage,numerical__Lot Area,numerical__Overall Qual,numerical__Overall Cond,numerical__Year Built,numerical__Year Remod/Add,numerical__Mas Vnr Area,...,categorical_other__Sale Type_New,categorical_other__Sale Type_Oth,categorical_other__Sale Type_VWD,categorical_other__Sale Type_WD,categorical_other__Sale Condition_Abnorml,categorical_other__Sale Condition_AdjLand,categorical_other__Sale Condition_Alloca,categorical_other__Sale Condition_Family,categorical_other__Sale Condition_Normal,categorical_other__Sale Condition_Partial
381,-1.284166,-0.991273,-0.871817,0.514642,0.033810,0.673941,-0.526415,0.181084,-0.381277,0.531409,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
834,-0.746674,1.018349,0.062906,-0.047047,2.307082,-0.766750,-0.526415,-0.115603,-0.814347,-0.569155,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1898,0.515781,-0.953797,0.763949,0.046568,-0.035514,-1.487095,-0.526415,-0.280430,-1.054941,-0.569155,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
678,-0.931771,-0.948378,0.763949,-0.421506,-0.363746,-1.487095,-0.526415,-0.708978,-1.632368,-0.569155,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
700,-0.905667,0.995206,3.100756,-0.281084,-0.310697,-1.487095,0.378216,-1.664971,-1.632368,-0.569155,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [93]:
X_train_transformed_df.shape

(2344, 316)

In [94]:
X_test_transformed_df = pd.DataFrame(
    test_array,
    columns=feature_names,
    index=X_test.index
)

X_test_transformed_df.shape

(586, 316)

In [95]:
assert (
    X_train_transformed_df.shape[1]
    == len(feature_names)
)

assert (
    X_test_transformed_df.shape[1]
    == len(feature_names)
)

assert (
    X_train_transformed_df.isna().sum().sum()
    == 0
)

assert (
    X_test_transformed_df.isna().sum().sum()
    == 0
)

print(
    "Final preprocessing validation passed."
)

Final preprocessing validation passed.


### Interpretation

The `ColumnTransformer` combines the three preprocessing pipelines into
a single reusable transformation system.

Numerical features are imputed and standardized.

Categorical features are separated according to the meaning of their
missing values, then one-hot encoded.

The complete preprocessor is fitted using training data only.

The fitted preprocessor is then used to transform both training and
test data.

The transformed training and test datasets contain the same number of
features and contain no missing or infinite values.

One-hot encoding increases the number of features because individual
categorical variables can produce multiple binary features.